# TMJDLC — Google Drive Edition

Run this notebook in Google Colab. It installs the `drive` branch, runs the test suite, mounts Google Drive, and starts the Telegram control bot. The bot supports Javtiful video/channel submissions, page and video selection, adjustable parallel downloads, and a separate Drive upload queue.

**Secrets:** add `TG_BOT_TOKEN` and `TG_ADMIN_IDS` in Colab's Secrets panel and enable notebook access. Optional tuning secrets are documented in the Telegram setup section. Never put credentials directly in notebook cells.

## 1. Clone / update the drive branch and install dependencies

In [ ]:
from pathlib import Path
import os

REPO_DIR = Path('/content/TMJDLC')
if REPO_DIR.exists():
    %cd /content/TMJDLC
    !git fetch origin drive
    !git checkout drive
    !git pull --ff-only origin drive
else:
    %cd /content
    !git clone --branch drive --single-branch https://github.com/Telebotfaroff/TMJDLC.git
    %cd /content/TMJDLC

!pip -q install -r requirements.txt
!pip -q install -e .
!python -m playwright install --with-deps chromium

## 2. Run project tests

In [ ]:
%cd /content/TMJDLC
!python -m pytest -q

## 3. Mount Google Drive and initialize the library

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
from tmjdlc.drive.manager import DriveManager
from tmjdlc.storage.json_store import JsonStore

DRIVE_ROOT_NAME = 'TMJDLC Library'
drive_manager = DriveManager.from_colab(root_name=DRIVE_ROOT_NAME)
drive_root_id = drive_manager.ensure_root()

# Keep the duplicate database on mounted Drive, never on ephemeral Colab storage.
LIBRARY_DIR = Path('/content/drive/MyDrive') / DRIVE_ROOT_NAME
LIBRARY_DIR.mkdir(parents=True, exist_ok=True)
database = JsonStore(LIBRARY_DIR / 'database.json')
WORK_DIR = Path('/content/TMJDLC-temp')
WORK_DIR.mkdir(parents=True, exist_ok=True)

print('Drive library ready:', LIBRARY_DIR)
print('Drive root ID:', drive_root_id)
print('Database:', database.path)

## 4. Download one Javtiful video and save it to Drive

Paste a single Javtiful post URL. Duplicate codes already recorded in `database.json` are skipped. A local job folder is deleted only after the Drive upload is verified and the database/index have been updated. If an upload or database step fails, the temporary file is retained for recovery.

In [ ]:
from tmjdlc.downloader.engine import DownloadEngine
from tmjdlc.providers.javtiful.provider import JavtifulURLParser

post_url = input('Javtiful video URL: ').strip()
parsed = JavtifulURLParser().parse(post_url)
if not parsed.code:
    raise ValueError('Could not identify a catalogue code from this URL. Check the post URL.')

code = parsed.code.upper()
if database.contains_code(code):
    print(f'SKIPPED: {code} is already present in the Drive library.')
else:
    def show_progress(data):
        status = data.get('status', 'working')
        if status == 'downloading':
            done = data.get('downloaded_bytes') or 0
            total = data.get('total_bytes')
            percent = f'{done / total * 100:.1f}%' if total else f'{done / 1024 / 1024:.1f} MiB'
            speed = data.get('speed')
            speed_text = f'{speed / 1024 / 1024:.2f} MiB/s' if speed else 'calculating'
            print(f'\\rDownloading {percent} | {speed_text}', end='')
        elif status == 'resolving':
            print(data.get('message', 'Resolving stream...'))
        elif status == 'finished':
            print('\\nDownload transfer finished; checking output...')

    engine = DownloadEngine(WORK_DIR, progress_callback=show_progress)
    result = engine.download(post_url)
    print('\\nDownloaded:', result.filename, f'({result.size / 1024 / 1024:.2f} MiB)')

    folder = drive_manager.ensure_code_folder(code)
    uploaded = drive_manager.upload_file(result.filepath, folder['id'])
    if int(uploaded.get('size', -1)) != result.size:
        raise IOError('Drive upload size verification failed; local file retained.')

    database.record_video(
        code,
        folder_id=folder['id'],
        folder_url=folder['url'],
        filename=result.filename,
        size=result.size,
    )
    current = database.read()
    index_entries = {
        key: value['folder_url']
        for key, value in current['videos'].items()
        if value.get('folder_url')
    }
    drive_manager.write_index(index_entries)

    # Delete temporary data only after upload, database write and index update succeed.
    engine.cleanup_job(result.job_id)
    print('SUCCESS:', code)
    print('Drive folder:', folder['url'])
    print('Uploaded file ID:', uploaded['id'])
    print('Local temporary file cleaned.')

## 5. Inspect the saved library

This cell displays completed codes from the persistent Drive database.

In [ ]:
library = database.read()
print('Completed videos:', len(library.get('videos', {})))
for video_code, item in sorted(library.get('videos', {}).items()):
    print(f"{video_code} | {item.get('filename')} | {item.get('size', 0) / 1024 / 1024:.2f} MiB | {item.get('folder_url')}")

## Notes

- Run the cells from top to bottom in a fresh Colab runtime.
- In Colab Secrets, add `TG_BOT_TOKEN` and `TG_ADMIN_IDS`, and enable Notebook access for both.
- Optional concurrency/logging values can also be stored as Colab Secrets; defaults are applied when omitted.
- Browser resolution requires Chromium; the install cell handles it.
- Signed stream URLs are short-lived and should not be copied into logs or saved in the database.
- `database.json` and its backup/lock files live in the mounted Drive library folder.
- The manual one-video cell is optional; the Telegram bot uses the same Drive manager and persistent database.

## 6. Configure Telegram using Colab Secrets

Before running the next cell, open the **Secrets** panel in Colab (key icon) and add these secrets:

- `TG_BOT_TOKEN` — your BotFather token (**required**)
- `TG_ADMIN_IDS` — your numeric Telegram user ID; multiple IDs can be comma-separated (**required**)
- `TMJDLC_MAX_PARALLEL_DOWNLOADS` — optional, default `2`, allowed `1`–`6`
- `TMJDLC_MAX_PARALLEL_UPLOADS` — optional, default `1`, allowed `1`–`4`
- `TMJDLC_LOG_LEVEL` — optional, default `INFO`

For each secret, switch **Notebook access** on. The token is read directly from Colab Secrets; it is not displayed or saved in this notebook.

In [ ]:
from google.colab import userdata
import os

# Required credentials are fetched from Colab Secrets. Nothing is prompted or printed.
os.environ['TG_BOT_TOKEN'] = userdata.get('TG_BOT_TOKEN').strip()
os.environ['TG_ADMIN_IDS'] = userdata.get('TG_ADMIN_IDS').strip()

# Keep the bot pointed at the same mounted Drive library and temporary workspace
# initialized above. Optional tuning values can also be stored as Colab Secrets.
os.environ['TMJDLC_DRIVE_ROOT_NAME'] = DRIVE_ROOT_NAME
os.environ['TMJDLC_WORK_DIR'] = str(WORK_DIR)
os.environ['TMJDLC_MAX_PARALLEL_DOWNLOADS'] = (userdata.get('TMJDLC_MAX_PARALLEL_DOWNLOADS') or '2').strip()
os.environ['TMJDLC_MAX_PARALLEL_UPLOADS'] = (userdata.get('TMJDLC_MAX_PARALLEL_UPLOADS') or '1').strip()
os.environ['TMJDLC_LOG_LEVEL'] = (userdata.get('TMJDLC_LOG_LEVEL') or 'INFO').strip()

if not os.environ['TG_BOT_TOKEN'] or not os.environ['TG_ADMIN_IDS']:
    raise RuntimeError('Add TG_BOT_TOKEN and TG_ADMIN_IDS in Colab Secrets, then enable Notebook access.')

print('Telegram credentials loaded from Colab Secrets. Values are hidden.')

## 7. Start the Telegram bot

Run the Secrets-loading cell first, then this cell. Keep the Colab runtime alive while the bot is working. In Telegram, send `/start`, then submit a Javtiful channel or video URL. The bot guides page selection, video selection and download concurrency. Use `/status` to inspect the queue.

In [ ]:
%cd /content/TMJDLC
from tmjdlc.config import Settings
from tmjdlc.bot.application import start_bot_async, stop_bot_async

bot_app = await start_bot_async(Settings.from_env(), drive_manager, database)
print('TMJDLC Telegram bot is running. Keep this Colab runtime alive.')

## Stop the Telegram bot

Run this cell when you want to stop polling cleanly.

In [ ]:
await stop_bot_async(bot_app)
print('TMJDLC bot stopped.')